# PCB 부품 검출 - Colab 학습 (base / rot)
런타임 > 런타임 유형 변경 > **T4 GPU** 를 먼저 선택하세요.

사전 준비: 구글 드라이브 `내 드라이브/pcb_eval/` 폴더에 **pcb_yolo.zip** 을 올려 두세요.

In [ ]:
!nvidia-smi -L
!pip -q install ultralytics

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 데이터 풀기 (train / val / test 폴더가 생김)
!mkdir -p /content/pcb_yolo
!unzip -q -o /content/drive/MyDrive/pcb_eval/pcb_yolo.zip -d /content/pcb_yolo
!ls /content/pcb_yolo
!ls /content/pcb_yolo/train/images | wc -l

In [ ]:
# Colab 경로용 data.yaml 새로 작성 (test 는 학습/검증에 쓰지 않음)
NAMES = ['Cap1','Cap2','Cap3','Cap4','MOSFET','Mov','Resistor','Transformer']
open('/content/pcb_yolo/data.yaml','w').write(
    'path: /content/pcb_yolo\ntrain: train/images\nval: val/images\n'
    f'nc: {len(NAMES)}\nnames: {NAMES}\n')
print(open('/content/pcb_yolo/data.yaml').read())

## 1) base 모델 - 회전 증강 없음 (보완 전)
결과는 드라이브(`pcb_eval/runs`)에 저장되므로 런타임이 끊겨도 남습니다.

In [ ]:
from ultralytics import YOLO
PROJECT = '/content/drive/MyDrive/pcb_eval/runs'

model = YOLO('yolov8n.pt')
model.train(data='/content/pcb_yolo/data.yaml', name='base', project=PROJECT,
            epochs=100, imgsz=640, batch=16, degrees=0, flipud=0.0, seed=0, workers=2)

## 2) rot 모델 - 회전 증강 (보완 후)
※ base 와 epochs / imgsz / seed 등은 같고 **회전 증강만** 다릅니다.

In [ ]:
model = YOLO('yolov8n.pt')
model.train(data='/content/pcb_yolo/data.yaml', name='rot', project=PROJECT,
            epochs=100, imgsz=640, batch=16, degrees=30, flipud=0.5, seed=0, workers=2)

## 3) 확인 (val 기준 - test 는 아직 보지 않음)

In [ ]:
for n in ['base', 'rot']:
    m = YOLO(f'{PROJECT}/{n}/weights/best.pt')
    r = m.val(data='/content/pcb_yolo/data.yaml', split='val', verbose=False)
    print(n, 'val mAP50 =', round(r.box.map50, 3), ' mAP50-95 =', round(r.box.map, 3))

## 4) 가중치 내려받기
드라이브 `pcb_eval/runs/base/weights/best.pt`, `runs/rot/weights/best.pt` 를 PC 의 `pcb_eval/runs/` 아래 같은 구조로 복사하세요.